In [1]:
getwd()

[1] "/mnt/gsdata/projects/other/Flux/EcoRes/EcoRes/scripts"

In [2]:
setwd("/mnt/gsdata/projects/panops/panops-data-registry/data/flux")

In [3]:
packages <- c("bigleaf", "purrr", "REddyProc", "stringr", "tidyr", "dplyr", "plyr", "broom", "lutz", "sf")

# Install packages not yet installed
installed_packages <- packages %in% rownames(installed.packages())
if (any(installed_packages == FALSE)) {
  install.packages(packages[!installed_packages])
}

invisible(lapply(packages, library, character.only = TRUE))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


------------------------------------------------------------------------------

You have loaded plyr after dplyr - this is likely to cause problems.
If you need functions from both plyr and dplyr, please load plyr first, then dplyr:
library(plyr); library(dplyr)

------------------------------------------------------------------------------


Attaching package: ‘plyr’


The following objects are masked from ‘package:dplyr’:

    arrange, count, desc, failwith, id, mutate, rename, summarise,
    summarize


The following object is masked from ‘package:purrr’:

    compact


Linking to GEOS 3.12.2, GDAL 3.9.2, PROJ 9.5.0; sf_use_s2() is TRUE



In [ ]:
TimeZoneCalculatorFLUXNET <- function(lat, lon) {
  message("Calculation of Time zone")

  tz_name <- tz_lookup_coords(lat, lon, method = "accurate", warn = FALSE)

  dfout <- tz_offset(
    as.POSIXct("2018-01-01 12:00:00", tz = tz_name),
    tz_name
  )

  TimeZone_h.n <- dfout$utc_offset_h
  return(TimeZone_h.n)
}


In [ ]:
# ++ Function to calculate the functional properties

EFPcalc<-function(dat.all){

  #-- Set the growing season filter as 30% of the seasonal amplitude of daily GPP
  
  GSfilt <- 0.3

  if(is.na(summary(dat.all$P)[4]) | (summary(dat.all$P)[4] == 0)){
    print("##### Warning: no Precipitation data!!! No precipitation Filter")
    dat.filtered <- filter.data(data.frame(dat.all), quality.control = TRUE, filter.growseas=TRUE, filter.precip=FALSE, 
                                GPP="GPP_NT",doy="doy",year="year",tGPP=GSfilt,
                                precip="P", tprecip=0.1,precip.hours=24,records.per.hour=2,
                                vars.qc=c("TA","H","LE", "NEE", "VPD"), quality.ext="_QC",good.quality=1)
    precipAvail <- "no"
  } 
  
  if(!is.na(summary(dat.all$P)[4]) & (summary(dat.all$P)[4] != 0)){
    print("#### Precipitation data available!!! Precipitation Filter Active")
    # -- Indicate if the data are hourly (record.per.hour <- 1) or half-hourly (record.per.hour <- 2)
    record.per.hour <- 2
    
    dat.filtered <- filter.data(data.frame(dat.all), quality.control = TRUE, filter.growseas=TRUE, filter.precip=TRUE, 
                                GPP="GPP_NT",doy="doy",year="year",tGPP=GSfilt,
                                precip="P", tprecip=0.1,precip.hours=24,records.per.hour=record.per.hour,
                                vars.qc=c("TA","H","LE", "NEE", "VPD"), quality.ext="_QC",good.quality=1)
    precipAvail <- "yes"
  }  
  
  # Filtering G1 and WUE, remove also low u* values (u* <= 0.2)
  
  print('....computing WUE Metrics for the site')
  
  # Water-use efficiency (WUE):
  #   WUE = GPP / ET
  # 
  # Water-use efficiency based on NEE (WUE_NEE):
  #   WUE_NEE = NEE / ET
  #
  # Inherent water-use efficiency (IWUE; Beer et al. 2009):
  #   IWUE = (GPP * VPD) / ET
  # 
  # Underlying water-use efficiency (uWUE; Zhou et al. 2014):
  #   uWUE= (GPP * sqrt(VPD)) / ET
  # 
  # All metrics are calculated based on the median of all values. E.g. WUE = median(GPP/ET,na.rm=TRUE)
  # 
  aaa<-subset(dat.filtered, SW_IN > 200 & USTAR > 0.2)
  
  EFPsOut<-WUE.metrics(aaa, GPP = "GPP_NT", NEE = "NEE", LE = "LE", VPD = "VPD_kPa",
                       Tair = "TA", constants = bigleaf.constants())
  
  # Calculate maximum Evapotranspiration (95th percetile of half hourly values)
  
  print('....computing maximum Evapotranspiration')
  
  EFPsOut$ETmax<-quantile(aaa$ET, 0.95, na.rm=T)

  # Availability of precipitation
  EFPsOut$precipAvail <- precipAvail
  
  # Calculation of surface conductance and stomatal slope, G1
  print('....computing maximum surface conductance (Gsmax) and stomatal slope (G1)')
  
  # Using only measured data
  # -- Removing NA for wind speed and calculating air pressure from elevation data (Elevation is needed)
  
  aaa<- aaa %>% drop_na(WS)
  
  # -- Calculation of aerodynamic conductance
  
  if (dim(aaa)[1] != 0){
    aaa$pressure<-pressure.from.elevation(elevation,aaa$TA,aaa$VPD_kPa)
    Ga <- aerodynamic.conductance(aaa,Tair = "TA", pressure = "pressure",
                                  wind = "WS", ustar = "USTAR", H = "H", 
                                  Rb_model="Thom_1972")[,"Ga_h"]
  }
  
  if (dim(aaa)[1] == 0){
    aaa<-subset(dat.filtered, SW_IN > 200 & USTAR > 0.2)
    aaa$pressure<-pressure.from.elevation(elevation,aaa$TA,aaa$VPD_kPa)
    Ga <- 1
  }
  
  if(all(is.na(aaa$G))){
    Gs <- surface.conductance(aaa,Ga=Ga, Tair = "TA", pressure = "pressure", Rn = "NETRAD",
                              G = NULL, S = NULL, VPD = "VPD_kPa", LE = "LE",  
                              missing.G.as.NA = FALSE, 
                              missing.S.as.NA = FALSE)
    EFPsOut$Gavail<-"no"
  }
  
  if(!all(is.na(aaa$G))){
    Gs <- surface.conductance(aaa,Ga=Ga, Tair = "TA", pressure = "pressure", Rn = "NETRAD",
                              G = "G", S = NULL, VPD = "VPD_kPa", LE = "LE",  
                              missing.G.as.NA = FALSE, 
                              missing.S.as.NA = FALSE)
    EFPsOut$Gavail<-"yes"
  }
  
  aaa$Gs_mol<-Gs$Gs_mol
  
  aaa<- aaa %>% drop_na(Gs_mol)
  aaa <- subset(aaa, VPD_kPa > 0)

  #-- Calculation of maximum surface conductance as the 90th percentile of half hourly Gs
  EFPsOut$GSmax<-quantile(na.omit(Gs$Gs_ms), 0.90)
  
  nmin <- 40 #Set as 40 the minimum number of good data points to calculate the functions
  
  if(dim(aaa)[1] >= nmin){  
    
    if(all(is.na(aaa$CO2))){
      #- Fix CO2 concentration to 400 ppm if not available
      aaa$CO2 <- 400
      ### Use robust regression to minimize influence of outliers in Gs                           
      mod_USO <- stomatal.slope(aaa,model="USO",GPP="GPP_NT",Gs="Gs_mol", Tair = "TA", pressure = "pressure", 
                                VPD = "VPD_kPa", Ca = "CO2",robust.nls=TRUE,nmin=nmin,fitg0=FALSE)
      
      EFPsOut$CO2avail<-"no"
      
    }
    
    if(!all(is.na(aaa$CO2))){
      
      ### Use robust regression to minimize influence of outliers in Gs                           
      mod_USO <- stomatal.slope(aaa,model="USO",GPP="GPP_NT",Gs="Gs_mol", Tair = "TA", pressure = "pressure", 
                                VPD = "VPD_kPa", Ca = "CO2",robust.nls=TRUE,nmin=nmin,fitg0=FALSE)
      
      EFPsOut$CO2avail<-"yes"
      
    }
    
    EFPsOut$g1<-tidy(mod_USO)$estimate
    EFPsOut$g1_stderr<-tidy(mod_USO)$std.error
  }
  
  if(dim(aaa)[1] < nmin){  
    EFPsOut$g1<-NA
    EFPsOut$g1_stderr<-NA
  }

  print('....computing Evaporative fraction for the sites and its amplitude')
  
  aaa$EF <- aaa$LE/(aaa$LE+aaa$H)
  EFPsOut$EF <- median(aaa$EF, na.rm = TRUE)
  EFPsOut$EFampl <- quantile(na.omit(aaa$EF), 0.75, na.rm = TRUE) - quantile(na.omit(aaa$EF), 0.25, na.rm = TRUE)
  
  ### -- print('....computing LRC parameters for the site')
  print('....computing LRC parameters for the site')
  
  myLRC<-function(datafilt){
    
    numNAN<-150
    #if(sum(is.na(datafilt$NEE)) >= numNAN) return(NA)
    #browser()
    if(sum(is.na(datafilt$NEE))/length(datafilt$NEE) >= 0.8) return(NA)
    
    if(sum(is.na(datafilt$NEE))/length(datafilt$NEE) < 0.8){
      #print(paste("...optimizing..", unique(datafilt$FiveDaySeq)))
      # browser()
      result = tryCatch({
        fitLRC<-light.response(datafilt, NEE = "NEE", Reco = "Reco", PPFD = "PPFD",
                               PPFD_ref = 2000)
        out<-tidy(fitLRC)$estimate[2]
        return(out)
      }, error = function(e) {
        return(NA)
        
      })
      
      
    }
  }
  
  #Filter again data out all data with QC > 1 (retaining good quality 0 and 1), growing season, no precip filter
  
  dat.filtered <- filter.data(data.frame(dat.all), quality.control = TRUE, filter.growseas=TRUE, filter.precip=FALSE, 
                              GPP="GPP_NT",doy="doy",year="year",tGPP=GSfilt,
                              precip="P", tprecip=0.1,precip.hours=24,records.per.hour=2,
                              vars.qc=c("TA","H","LE", "NEE", "VPD"), quality.ext="_QC",good.quality=1)
  
  
  dat.filtered$FiveDaySeq<-rep(c(1:ceiling(dim(dat.filtered)[1]/5)),each = 48*5, length.out=dim(dat.filtered)[1])
  
  zzz<-data.frame(NEE=dat.filtered$NEE,
                  PPFD=dat.filtered$PPFD_IN_FROM_SWIN,
                  Reco=dat.filtered$RECO_NT,
                  FiveDaySeq = dat.filtered$FiveDaySeq)
  
  outGPPsat <- unlist(by(zzz, zzz$FiveDaySeq, myLRC), use.names=FALSE)
  
  EFPsOut$GPPsat<-quantile(outGPPsat, 0.90, na.rm = T)  
  EFPsOut$NEPmax<-quantile(na.omit(subset(zzz, PPFD > 200*2.11)$NEE*-1), 0.99)
  
  ### -- print('....computing Basal respiration with REddyProc')
  print('....computing Rb parameters for the site')
  
  ttt<-data.frame(dat.all)
  ttt$DateTime<-ttt$DateTime + 30*30
  TimeZone <- TimeZoneCalculatorFLUXNET(lat, lon)
  
  EddyProc.C <- sEddyProc$new(site, ttt, c('NEE', 'NEE_QC_OK', 'SW_IN', 'SW_IN_QC_OK',
                                           'TA','TA_QC_OK'),LatDeg=as.numeric(lat), LongDeg=as.numeric(lon),TimeZoneHour=TimeZone)
  
  #-- Run the nighttime partitioning for the calculation  of basal respiration
  
  print('Run the nighttime partitioning for the calculation  of basal respiration')
  
  EddyProc.C$sMRFluxPartition(FluxVar.s = "NEE", QFFluxVar.s = "NEE_QC_OK", 
                              QFFluxValue.n = 1, TempVar.s = "TA", QFTempVar.s = "TA_QC_OK", 
                              QFTempValue.n = 1, RadVar.s = "SW_IN", T_ref.n = 273.15 + 15, 
                              Suffix.s = "")
  
  EFPsOut$Rb<-mean(EddyProc.C$sTEMP$R_ref, na.rm = T)
  EFPsOut$Rbmax<-quantile(EddyProc.C$sTEMP$R_ref, 0.95)

  #-- Apparent Carbon Use Efficiency
  print('Calculation of apparent Carbon Use Efficiency, aCUE')
  
  dat.all$Rb <- ifelse(is.null(EddyProc.C$sTEMP$R_ref), NA, EddyProc.C$sTEMP$R_ref)
  
  dat.filtered.CUE <- filter.data(data.frame(dat.all), quality.control = TRUE, filter.growseas=TRUE, filter.precip=FALSE, 
                                  GPP="GPP_NT",doy="doy",year="year",tGPP=GSfilt,
                                  precip="P", tprecip=0.1,precip.hours=24,records.per.hour=2,
                                  vars.qc=c("TA","H","LE", "NEE", "VPD"), quality.ext="_QC",good.quality=1)

  daily.aggr<-ddply(dat.filtered.CUE, .(year, doy), summarize, mean_GPP=mean(GPP_NT), mean_Rb=mean(Rb))
  
  EFPsOut$aCUE <- median(1-(daily.aggr$mean_Rb/daily.aggr$mean_GPP), na.rm=TRUE)
  EFPsOut$TZ<-as.numeric(TimeZone)
  EFPsOut$nyears<-nyears
  EFPsOut$SITE_ID <- site
  
  EFPsF15LT<-unlist(EFPsOut)
  
  names(EFPsF15LT) <- str_replace_all(names(EFPsF15LT),c("ETmax.95%" = "ETmax",
                                     "GSmax.90%" = "GSmax",
                                     "g1" = "G1",
                                     "EFampl.75%" = "EFampl",  
                                     "GPPsat.90%" = "GPPsat",
                                     "NEPmax.99%" = "NEPmax",
                                     "Rbmax.95%" = "Rbmax"))
  
  EFPsF15LT <- data.frame(t(EFPsF15LT))
  output.EFPs <- subset(EFPsF15LT, select = c("uWUE","ETmax","precipAvail","Gavail","GSmax","CO2avail","G1",
                              "EF","EFampl","GPPsat","NEPmax","Rb","Rbmax","aCUE","TZ","nyears","SITE_ID"))
  
  print(paste0("...Finish site ",site))
  
  return(output.EFPs)
}

In [6]:
library(data.table)# Create a safe version of EFPcalc to avoid breaking the loop
tower_meta <- fread("combined_site_metadata.csv")


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose




In [7]:
tower_flux_folder <- "merged_sites_subset_renamed"

In [8]:
safeEFPcalc <- purrr::possibly(EFPcalc, otherwise = NA)

In [11]:
tower_flux_files <- list.files(
  path = tower_flux_folder,
  pattern = "\\.csv$",
  full.names = TRUE
)

file_lookup <- setNames(
  as.list(tower_flux_files),
  stringr::str_remove(
    tools::file_path_sans_ext(basename(tower_flux_files)),
    "_merged$"
  )
)


In [ ]:
# Folder where per-site EFP CSVs will be written
out_dir <- "efp_site_year_results"
dir.create(out_dir, showWarnings = FALSE)

#  keep a small list of results in memory 
efp_results_list <- list()

# Loop through each unique SITE_ID in tower_meta
for (this_site in unique(tower_meta$SITE_ID)) {
  message("Processing site: ", this_site)
  
  # --- Site metadata ----------------------------------------------------
  site_info <- tower_meta %>% filter(SITE_ID == this_site)
  
  # Skip if location info is missing
  if (any(
    is.na(site_info$LOCATION_ELEV),
    is.na(site_info$LOCATION_LAT),
    is.na(site_info$LOCATION_LONG)
  )) {
    warning("Missing location info for site ", this_site, "; skipping.")
    next
  }
  
  # Convert site metadata
  site      <- site_info$SITE_ID
  elevation <- as.numeric(site_info$LOCATION_ELEV)
  lat       <- as.numeric(site_info$LOCATION_LAT)
  lon       <- as.numeric(site_info$LOCATION_LONG)
  
  # Time zone (if used inside your EFP workflow) – silenced to avoid spam
  TimeZone <- tryCatch(
  suppressWarnings(TimeZoneCalculatorFLUXNET(lat = lat, lon = lon)),
  error = function(e) {
    warning("Time zone calculation failed for site ", this_site,
            ". Using 'UTC' instead.")
    0  # offset hours for UTC
  }
)
  
  # --- Read flux data for this site from per-site CSV -------------------
  flux_path <- file_lookup[[this_site]]
  
  if (is.null(flux_path) || !file.exists(flux_path)) {
    warning("No merged flux file found for site ", this_site, "; skipping.")
    next
  }
  
  flux_df <- data.table::fread(flux_path)
  
  if (nrow(flux_df) == 0) {
    warning("No data for site ", this_site, "; skipping.")
    next
  }
  
  # --- Date/time + derived variables ------------------------------------
  flux_df <- flux_df %>%
    mutate(
      DateTime = lubridate::ymd_hm(as.character(TIMESTAMP_START), tz = "UTC") +
        lubridate::minutes(15),
      year  = lubridate::year(DateTime),
      doy   = lubridate::yday(DateTime),
      ET    = LE.to.ET(LE, TA) * 1800,
      precip_mm          = P,
      VPD_kPa            = VPD / 10,
      PPFD_IN_FROM_SWIN  = SW_IN * 2.11,
      NEE_QC_OK   = NEE_QC,
      SW_IN_QC_OK = SW_IN_QC,
      TA_QC_OK    = TA_QC,
      SITE_ID = this_site
    )

  # *** IMPORTANT: define nyears for EFPcalc ***
  nyears <- dplyr::n_distinct(flux_df$year)

  # --- Year info --------------------------------------------------------
  years <- sort(unique(flux_df$year[!is.na(flux_df$year)]))
  
  if (length(site_year_results) > 0) {
    site_efp <- dplyr::bind_rows(site_year_results)

    site_meta_row <- site_info %>%
      dplyr::select(
        SITE_ID,
        LOCATION_LAT,
        LOCATION_LONG,
        LOCATION_ELEV,
        dplyr::any_of("IGBP")
      )

    site_efp <- dplyr::left_join(site_efp, site_meta_row, by = "SITE_ID")

    out_path <- file.path(out_dir, paste0(this_site, "_EFP_yearly.csv"))
    message("  → Writing file: ", out_path)
    readr::write_csv(site_efp, out_path)

    efp_results_list[[this_site]] <- site_efp
  } else {
    warning("No yearly EFP results to save for site ", this_site)
  }
  
  # If we got any yearly results for this site, combine + add metadata + write CSV
  if (length(site_year_results) > 0) {
    site_efp <- dplyr::bind_rows(site_year_results)
    
    # keep only the relevant metadata columns to join
    site_meta_row <- site_info %>%
      dplyr::select(
        SITE_ID,
        LOCATION_LAT,
        LOCATION_LONG,
        LOCATION_ELEV,
        dplyr::any_of("IGBP")
      )
    
    site_efp <- dplyr::left_join(site_efp, site_meta_row, by = "SITE_ID")
    
    # save one CSV per site with all its years
    out_path <- file.path(out_dir, paste0(this_site, "_EFP_yearly.csv"))
    readr::write_csv(site_efp, out_path)
    
    # (optional) also keep in memory
    efp_results_list[[this_site]] <- site_efp
  } else {
    warning("No yearly EFP results to save for site ", this_site)
  }
}

Processing site: AT-Neu



In [ ]:
list.files("efp_site_year_results")